# Notebook 03_99 — Merge Feature Engineering Results

Combines the 5 per-method CSVs (`results/fe_*_raw.csv`) produced by notebooks
`03_01` .. `03_05`, then produces:
- Mean ± std summary table
- Best FE method per classifier
- F1 heatmaps (method × K, per classifier)
- FS vs FE comparison plot
- Wilcoxon significance test vs the notebook-01 baseline

In [1]:
import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CODE_DIR = '/content/UAV' if IN_COLAB else os.environ.get('UAV_CODE_DIR', 'D:/UAV_')

if IN_COLAB and not os.path.isdir(os.path.join(CODE_DIR, '.git')):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/haribawngg/UAV.git', CODE_DIR], check=True)

sys.path.insert(0, os.path.join(CODE_DIR, 'notebook'))
from common import *
from scipy.stats import wilcoxon
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

print('Imports OK')

CPU: 16 logical cores detected -> N_JOBS=-1 (RF/XGB/LGBM/KNN use all cores)


Imports OK


## 1. Load and Concatenate All FE Results

In [2]:
d = load_data()
CLASSIFIER_NAMES = d['classifier_names']

FE_FILES = {
    'PCA'         : 'fe_pca_raw.csv',
    'LDA'         : 'fe_lda_raw.csv',
    'KernelPCA'   : 'fe_kernelpca_raw.csv',
    'AutoEncoder' : 'fe_autoencoder_raw.csv',
    'StatFeatures': 'fe_statfeatures_raw.csv',
}

dfs = []
for method, fname in FE_FILES.items():
    path = f'{RESULTS_DIR}{fname}'
    if not os.path.exists(path):
        print(f'MISSING: {path}  -> run the corresponding 03_0X notebook first.')
        continue
    df = pd.read_csv(path)
    expected = len(K_VALUES) * len(SEEDS) * len(CLASSIFIER_NAMES)
    status = 'COMPLETE' if len(df) == expected else f'PARTIAL ({len(df)}/{expected})'
    print(f'{method:<14} {fname:<28} rows={len(df):<5} {status}')
    dfs.append(df)

fe_raw = pd.concat(dfs, ignore_index=True)
print(f'\nTotal combined rows: {len(fe_raw)}')

PCA            fe_pca_raw.csv               rows=90    COMPLETE
LDA            fe_lda_raw.csv               rows=90    COMPLETE
KernelPCA      fe_kernelpca_raw.csv         rows=90    COMPLETE
AutoEncoder    fe_autoencoder_raw.csv       rows=90    COMPLETE
StatFeatures   fe_statfeatures_raw.csv      rows=90    COMPLETE

Total combined rows: 450


## 2. Mean ± Std Summary Table

In [3]:
rows = []
for method in fe_raw['method'].unique():
    for K in sorted(fe_raw['K'].unique()):
        for clf in CLASSIFIER_NAMES:
            sub = fe_raw[(fe_raw['method'] == method) & (fe_raw['K'] == K) & (fe_raw['classifier'] == clf)]
            if sub.empty:
                continue
            row = {'FE_Method': method, 'K': K, 'actual_K': sub['actual_K'].iloc[0], 'Classifier': clf}
            for m in METRICS:
                row[m] = f"{sub[m].mean():.4f} ± {sub[m].std(ddof=1):.4f}"
            rows.append(row)

fe_summary = pd.DataFrame(rows)
fe_summary.to_csv(f'{RESULTS_DIR}fe_results_summary.csv', index=False)
print(fe_summary.head(30).to_string(index=False))
print(f'\nSaved: {RESULTS_DIR}fe_results_summary.csv')

FE_Method  K  actual_K Classifier        accuracy       precision          recall              f1          pr_auc             fpr             fnr      train_time_s    inference_ms
      PCA  4         4         DT 0.9941 ± 0.0003 0.8920 ± 0.0012 0.8982 ± 0.0006 0.8909 ± 0.0013 0.8949 ± 0.0012 0.0006 ± 0.0000 0.1018 ± 0.0006   0.1890 ± 0.0071 0.0001 ± 0.0000
      PCA  4         4         RF 0.9941 ± 0.0003 0.8907 ± 0.0018 0.8983 ± 0.0002 0.8907 ± 0.0012 0.8992 ± 0.0001 0.0006 ± 0.0000 0.1017 ± 0.0002   1.9270 ± 0.1206 0.0027 ± 0.0000
      PCA  4         4        XGB 0.9941 ± 0.0003 0.8974 ± 0.0044 0.8988 ± 0.0005 0.8923 ± 0.0008 0.8995 ± 0.0002 0.0006 ± 0.0000 0.1012 ± 0.0005   1.7113 ± 0.0887 0.0010 ± 0.0000
      PCA  4         4       LGBM 0.8478 ± 0.3270 0.7817 ± 0.2496 0.8399 ± 0.1304 0.7813 ± 0.2474 0.7889 ± 0.2471 0.0155 ± 0.0334 0.1601 ± 0.1304   1.1008 ± 0.0792 0.0027 ± 0.0009
      PCA  4         4        KNN 0.9632 ± 0.0000 0.8356 ± 0.0108 0.7760 ± 0.0002 0.7530 ± 0.0055 0.

## 3. Best FE Method Per Classifier

In [4]:
print('=== BEST FE METHOD PER CLASSIFIER (highest mean F1) ===')
best_rows = []
for clf in CLASSIFIER_NAMES:
    sub = fe_raw[fe_raw['classifier'] == clf]
    grouped = sub.groupby(['method', 'K'])['f1'].mean().reset_index()
    if grouped.empty:
        continue
    best = grouped.loc[grouped['f1'].idxmax()]
    train_t = sub[(sub['method'] == best['method']) & (sub['K'] == best['K'])]['train_time_s'].mean()
    print(f"  {clf:6s}  best FE={best['method']:<14} K={int(best['K'])}  F1={best['f1']:.4f}  train={train_t:.1f}s")
    best_rows.append({'Classifier': clf, 'Best_FE': best['method'], 'K': int(best['K']),
                       'F1': best['f1'], 'train_time_s': train_t})

pd.DataFrame(best_rows).to_csv(f'{RESULTS_DIR}fe_best_per_classifier.csv', index=False)

=== BEST FE METHOD PER CLASSIFIER (highest mean F1) ===
  DT      best FE=PCA            K=8  F1=0.8931  train=0.3s
  RF      best FE=PCA            K=8  F1=0.8930  train=1.9s
  XGB     best FE=PCA            K=8  F1=0.8924  train=1.8s
  LGBM    best FE=PCA            K=8  F1=0.8925  train=1.4s
  KNN     best FE=AutoEncoder    K=8  F1=0.7574  train=0.2s
  MLP     best FE=LDA            K=8  F1=0.8913  train=29.7s


## 4. Heatmaps — F1 by Method × K (per classifier)

In [5]:
FE_METHODS = list(FE_FILES.keys())
K_present  = sorted(fe_raw['K'].unique())
n_clf = len(CLASSIFIER_NAMES)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, clf in zip(axes, CLASSIFIER_NAMES):
    matrix = np.full((len(FE_METHODS), len(K_present)), np.nan)
    for i, fe in enumerate(FE_METHODS):
        for j, K in enumerate(K_present):
            sub = fe_raw[(fe_raw['method'] == fe) & (fe_raw['K'] == K) & (fe_raw['classifier'] == clf)]
            if not sub.empty:
                matrix[i, j] = sub['f1'].mean()
    sns.heatmap(matrix, annot=True, fmt='.3f', cmap='YlOrRd',
                xticklabels=[f'K={k}' for k in K_present],
                yticklabels=FE_METHODS, ax=ax, vmin=0, vmax=1)
    ax.set_title(f'{clf} — Macro F1')

for ax in axes[n_clf:]:
    ax.set_visible(False)

plt.suptitle('Feature Engineering: Macro F1 by Method × K (mean over completed seeds)', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}fe_f1_heatmap.png', dpi=100, bbox_inches='tight')
plt.close()
print(f'Saved: {RESULTS_DIR}fe_f1_heatmap.png')

Saved: D:/UAV_/results/fe_f1_heatmap.png


## 5. FS vs FE Comparison (requires 02_99 to have been run)

In [6]:
fs_summary_path = f'{RESULTS_DIR}fs_results_summary.csv'
if not os.path.exists(fs_summary_path):
    print('FS summary not found — run 02_99_fs_merge_results.ipynb first to enable this comparison.')
else:
    fs_df = pd.read_csv(fs_summary_path)

    def parse_mean(s):
        return float(str(s).split('±')[0].strip())

    fs_df['f1_mean'] = fs_df['f1'].apply(parse_mean)
    fe_df_summary = pd.read_csv(f'{RESULTS_DIR}fe_results_summary.csv')
    fe_df_summary['f1_mean'] = fe_df_summary['f1'].apply(parse_mean)

    fig, axes = plt.subplots(1, n_clf, figsize=(5 * n_clf, 5))
    if n_clf == 1:
        axes = [axes]

    for ax, clf in zip(axes, CLASSIFIER_NAMES):
        fs_sub = fs_df[fs_df['Classifier'] == clf].groupby('K')['f1_mean'].max()
        fe_sub = fe_df_summary[fe_df_summary['Classifier'] == clf].groupby('K')['f1_mean'].max()

        for K in K_present:
            fs_k = fs_sub.get(K, np.nan)
            fe_k = fe_sub.get(K, np.nan)
            ax.scatter(K, fs_k, marker='o', color='steelblue', s=80, label='FS (best)' if K == K_present[0] else '')
            ax.scatter(K, fe_k, marker='s', color='tomato', s=80, label='FE (best)' if K == K_present[0] else '')

        ax.set_title(clf)
        ax.set_xlabel('K')
        ax.set_ylabel('Macro F1')
        ax.set_xticks(K_present)
        ax.legend()

    plt.suptitle('Best FS vs Best FE — Macro F1 per Classifier and K')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}fs_vs_fe_comparison.png', dpi=100)
    plt.close()
    print(f'Saved: {RESULTS_DIR}fs_vs_fe_comparison.png')

Saved: D:/UAV_/results/fs_vs_fe_comparison.png


## 6. Statistical Significance vs Baseline (Wilcoxon)

In [7]:
baseline_path = f'{RESULTS_DIR}baseline_raw_scores.json'
if not os.path.exists(baseline_path):
    print('Baseline scores not found — run 01_baseline.ipynb first.')
else:
    with open(baseline_path) as f:
        baseline_scores = json.load(f)

    sig_rows = []
    for clf in CLASSIFIER_NAMES:
        base_f1 = baseline_scores[clf]['f1']
        for fe in FE_METHODS:
            for K in K_present:
                sub = fe_raw[(fe_raw['method'] == fe) & (fe_raw['K'] == K) & (fe_raw['classifier'] == clf)]
                if len(sub) != len(SEEDS):
                    continue
                fe_f1 = sub.sort_values('seed')['f1'].tolist()
                try:
                    _, p = wilcoxon(fe_f1, base_f1, alternative='two-sided')
                except Exception:
                    p = float('nan')
                sig_rows.append({
                    'Classifier': clf, 'FE': fe, 'K': K,
                    'FE_F1_mean': np.mean(fe_f1), 'Baseline_F1_mean': np.mean(base_f1),
                    'p_value': p, 'significant (alpha=0.05)': 'Yes' if p < 0.05 else 'No'
                })

    sig_df = pd.DataFrame(sig_rows)
    sig_df.to_csv(f'{RESULTS_DIR}fe_significance.csv', index=False)
    print(f'Saved: {RESULTS_DIR}fe_significance.csv  ({len(sig_df)} rows)')
    print(sig_df.head(20).to_string(index=False))

Saved: D:/UAV_/results/fe_significance.csv  (90 rows)
Classifier           FE  K  FE_F1_mean  Baseline_F1_mean  p_value significant (alpha=0.05)
        DT          PCA  4    0.890885          0.890651   0.6250                       No
        DT          PCA  8    0.893052          0.890651   0.0625                       No
        DT          PCA 16    0.889852          0.890651   0.1250                       No
        DT          LDA  4    0.891611          0.890651   0.0625                       No
        DT          LDA  8    0.891640          0.890651   0.0625                       No
        DT          LDA 16    0.889229          0.890651   0.6250                       No
        DT    KernelPCA  4    0.853546          0.890651   0.0625                       No
        DT    KernelPCA  8    0.815761          0.890651   0.0625                       No
        DT    KernelPCA 16    0.815313          0.890651   0.0625                       No
        DT  AutoEncoder  4    0.8462

## 7. Completion Check

In [8]:
expected_total = len(FE_FILES) * len(K_VALUES) * len(SEEDS) * len(CLASSIFIER_NAMES)
print(f'Expected total rows : {expected_total}')
print(f'Actual total rows   : {len(fe_raw)}')
if len(fe_raw) < expected_total:
    print('\n⚠ Some experiments are missing. Re-run the corresponding 03_0X notebook(s) — they will resume automatically.')
else:
    print('\n✓ All FE experiments complete.')

print('\nNote: LDA and StatFeatures clip K=16 to their hard maximum (9 and 10 respectively).')
print('Check the actual_K column when interpreting K=16 results for those two methods.')

Expected total rows : 450
Actual total rows   : 450

✓ All FE experiments complete.

Note: LDA and StatFeatures clip K=16 to their hard maximum (9 and 10 respectively).
Check the actual_K column when interpreting K=16 results for those two methods.
